# eisfit demo

This notebook demonstrates fitting a battery EIS equivalent-circuit model using the public `EIS_dataset/ID01.csv` example.

Base model:

```text
Z = jωL + R0 + (R1 || CPE1) + (R2 || CPE2) + tail
```

Available low-frequency tails:

```text
CPE tail, default:       tail = 1 / (Qd * (jω)^nd)
Warburg tail:            tail = σ / sqrt(jω)
```

The Warburg option is the traditional semi-infinite Warburg form; the default CPE option keeps the exponent free.

Dataset citation: M. Moertelmaier, M. Kasper, and S. Clark, *EIS data of 54 21700 cells*, Zenodo, 2025, doi: 10.5281/zenodo.15422339. The included dataset metadata states CC-BY 4.0 licensing.

## Computational environment

The following cell imports the numerical, tabular, and plotting libraries required for the analysis. It also resolves the local project root so that the development version of `eisfit` can be imported reproducibly when the notebook is executed from either the repository root or the `examples` directory.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / "src" / "eisfit").exists():
        project_root = candidate
        break
else:
    project_root = Path("..").resolve()

src_path = project_root / "src"
sys.path = [path for path in sys.path if Path(path or ".").resolve() != src_path]
sys.path.insert(0, str(src_path))

for module_name in list(sys.modules):
    if module_name == "eisfit" or module_name.startswith("eisfit."):
        del sys.modules[module_name]

from eisfit import fit, load_eis

## Data loading

This cell loads the impedance spectrum for cell `ID01` from the Zenodo-derived example dataset. The returned frequency vector and complex impedance array are inspected briefly to confirm the number of observations and the numerical format used in subsequent model fitting.

In [ ]:
data_path = project_root / "EIS_dataset" / "ID01.csv"
freq, Zexp = load_eis(data_path)
len(freq), freq[:3], Zexp[:3]

## Baseline fit with the default CPE tail

The following cell estimates the equivalent-circuit parameters using the default low-frequency constant-phase-element tail. Multiple random starts are used to reduce sensitivity to local minima, while the fixed seed makes the demonstration reproducible.

In [ ]:
# Default model: CPE tail
# tail="cpe" is the default, so users can call fit(freq, Zexp) directly.
default_cpe = fit(freq, Zexp, n_starts=20, seed=7, max_nfev=5000)

pd.DataFrame({'Parameter': default_cpe.parameters.keys(), 'Value': default_cpe.parameters.values()})

## Comparison with a semi-infinite Warburg tail

This cell fits an otherwise identical circuit in which the low-frequency tail is represented by the traditional semi-infinite Warburg element. The resulting table reports both goodness-of-fit metrics and fitted parameters, enabling a direct comparison between the flexible CPE tail and the constrained Warburg formulation.

In [ ]:
# Alternative model: traditional semi-infinite Warburg tail
warburg = fit(freq, Zexp, tail="warburg", n_starts=20, seed=7, max_nfev=5000)

comparison = pd.DataFrame([
    {"Model": "Default CPE tail", "Tail": default_cpe.tail, "RMSE_ohm": default_cpe.rmse, "NRMSE": default_cpe.nrmse, **default_cpe.parameters},
    {"Model": "Warburg tail", "Tail": warburg.tail, "RMSE_ohm": warburg.rmse, "NRMSE": warburg.nrmse, **warburg.parameters},
])
comparison

## Visual assessment of model agreement

The final cell visualizes the experimental spectrum together with both fitted model responses. Nyquist, phase, and real-impedance representations are shown side by side because each view emphasizes different aspects of the fit quality across the measured frequency range.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)

axes[0].plot(Zexp.real, -Zexp.imag, "o", ms=3, label="Experiment")
axes[0].plot(default_cpe.Zsim.real, -default_cpe.Zsim.imag, "-", lw=2, label="Default CPE tail")
axes[0].plot(warburg.Zsim.real, -warburg.Zsim.imag, "--", lw=2, label="Warburg tail")
axes[0].set_xlabel("Z' (Ω)")
axes[0].set_ylabel("-Z'' (Ω)")
axes[0].set_title("Nyquist")

phase_exp = -np.angle(Zexp, deg=True)
phase_cpe = -np.angle(default_cpe.Zsim, deg=True)
phase_warburg = -np.angle(warburg.Zsim, deg=True)

axes[1].semilogx(freq, phase_exp, "o", ms=3, label="Experiment")
axes[1].semilogx(freq, phase_cpe, "-", lw=2, label="Default CPE tail")
axes[1].semilogx(freq, phase_warburg, "--", lw=2, label="Warburg tail")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("-Phase (deg)")
axes[1].set_title("Phase")

axes[2].semilogx(freq, Zexp.real, "o", ms=3, label="Experiment")
axes[2].semilogx(freq, default_cpe.Zsim.real, "-", lw=2, label="Default CPE tail")
axes[2].semilogx(freq, warburg.Zsim.real, "--", lw=2, label="Warburg tail")
axes[2].set_xlabel("Frequency (Hz)")
axes[2].set_ylabel("Z' (Ω)")
axes[2].set_title("Real impedance")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False)

fig